# Fine-tuning — LightGBM (transaction-only)

Goal: squeeze more AUC out of the **transaction-only** LightGBM baseline
(`baseline_lgbm`, 0.9006) by tuning its hyperparameters — without re-including
`identity` (the raw join didn't help).

Key design choices so this stays honest & comparable:

- **Same data & final split as `Modeling.ipynb`** — same parquet, same
  `temporal_split` (last 20% of time by `TransactionDT`). The final
  `X_val`/`y_val` are **never** used during the search, only for the final
  evaluation, so the reported number is directly comparable to `baseline_lgbm`.
- **Time-series CV inside the training window** — hyperparameters are chosen
  with `TimeSeriesSplit` on `X_train` only (chronological folds, no random
  KFold → no temporal leakage).
- **Separate MLflow experiment** (`ieee-fraud-detection-finetune`) so tuning
  runs don't pollute the baseline comparison in `Modeling.ipynb`.

> **Approach (decided):** randomized search via sklearn `ParameterSampler`,
> `N_TRIALS = 30`, 5-fold `TimeSeriesSplit` with per-fold early stopping; only
> the final best model is logged to MLflow. No new dependencies (`optuna` is
> not used).

In [1]:
import numpy as np
import pandas as pd

import mlflow
import lightgbm as lgb
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit

from ieee_cis_fraud_detection.config import PROCESSED_DATA_DIR, PROJ_ROOT

transaction = pd.read_parquet(PROCESSED_DATA_DIR / "train_transaction_filtered.parquet")
print("train_transaction:", transaction.shape)

TARGET = "isFraud"
DROP_COLS = ["TransactionID"]  # pure ID column, never a feature

/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-27 06:37:04.774 | INFO     | ieee_cis_fraud_detection.config:<module>:11 - PROJ_ROOT path is: /Users/alex/IEEE-CIS_Fraud_Detection_MLOp


train_transaction: (590540, 220)


In [2]:
def prepare_data(df: pd.DataFrame):
    """Separate features and labels (same as Modeling.ipynb)."""
    y = df[TARGET].astype(int).to_numpy()
    X = df.drop(columns=[TARGET] + DROP_COLS)
    return X, y


def temporal_split(df: pd.DataFrame, val_frac: float = 0.2):
    """Time-ordered holdout: the last `val_frac` of time goes to validation."""
    df = df.sort_values("TransactionDT").reset_index(drop=True)
    split_idx = int(len(df) * (1 - val_frac))
    return df.iloc[:split_idx], df.iloc[split_idx:]


train_df, val_df = temporal_split(transaction, val_frac=0.2)
X_train, y_train = prepare_data(train_df)
X_val, y_val = prepare_data(val_df)

print(f"train: {X_train.shape}  val: {X_val.shape}")
print(f"val fraud rate: {y_val.mean():.4f}")

train: (472432, 218)  val: (118108, 218)
val fraud rate: 0.0344


## Tuning strategy (no leakage)

- **Search-time validation**: `TimeSeriesSplit` over `X_train` only (chronological
  folds). Each trial reports the **mean validation AUC** across folds.
- **Final evaluation**: the best parameters are retrained on the **full**
  `X_train` and scored on the held-out `X_val` — the same validation set that
  produced `baseline_lgbm`'s 0.9006. This final number is what we compare.

In [3]:
# --- MLflow setup ---------------------------------------------------------
# Separate experiment so tuning runs stay out of the baseline compare in
# Modeling.ipynb.
TRACKING_DB = PROJ_ROOT / "mlruns" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{TRACKING_DB}")
mlflow.set_experiment("ieee-fraud-detection-finetune")

# We only log the FINAL best model (`finetuned_lgbm`) — the search itself is
# NOT logged, so MLflow stays clean. (No mlflow.autolog() here, which would
# create noisy nested runs for every CV fit.)

2026/08/27 06:37:10 INFO mlflow.tracking.fluent: Experiment with name 'ieee-fraud-detection-finetune' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/notebooks/mlruns/2', creation_time=1787783830792, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1787783830792, lifecycle_stage='active', name='ieee-fraud-detection-finetune', tags={}, trace_location=None, workspace='default'>

## Randomized search (sklearn)

We draw `N_TRIALS = 30` hyperparameter configs from the distribution space below
via sklearn's `ParameterSampler` — the same random-search engine behind
`RandomizedSearchCV`.

> **Why not `RandomizedSearchCV` directly?** Its cross-validation only passes
> each fold's *training* slice to `fit()`, so it can't supply an `eval_set` —
> which LightGBM needs for early stopping. This small loop is functionally a
> randomized search, but with **per-fold early stopping** (5 chronological
> folds via `TimeSeriesSplit`).

In [4]:
N_TRIALS = 30        # number of random configs to try
N_CV_SPLITS = 5      # chronological folds (TimeSeriesSplit) on X_train
EARLY_STOPPING_ROUNDS = 50
MAX_TREES = 2000     # upper cap; early stopping decides the actual tree count

# Sampling space (scipy distributions -> ParameterSampler).
param_dist = {
    "learning_rate": loguniform(0.01, 0.1),
    "num_leaves": randint(8, 256),
    "min_child_samples": randint(10, 201),
    "subsample": uniform(0.5, 0.5),         # [0.5, 1.0]
    "colsample_bytree": uniform(0.4, 0.6),  # [0.4, 1.0]
    "reg_alpha": loguniform(1e-4, 10.0),
    "reg_lambda": loguniform(1e-4, 10.0),
}


def evaluate_params(params: dict) -> tuple[float, float]:
    """TimeSeriesSplit CV with early stopping.

    Returns (mean CV AUC, mean best_iteration) — the latter is the tree budget
    used later for the final fit.
    """
    tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
    aucs, iters = [], []
    for tr_idx, va_idx in tscv.split(X_train):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train.iloc[tr_idx], y_train[tr_idx],
            eval_set=[(X_train.iloc[va_idx], y_train[va_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        y_score = model.predict_proba(X_train.iloc[va_idx])[:, 1]
        aucs.append(roc_auc_score(y_train[va_idx], y_score=y_score))
        iters.append(model.best_iteration_)
    return float(np.mean(aucs)), float(np.mean(iters))

In [5]:
# --- Run the randomized search --------------------------------------------
results = []
for trial_id, sampled in enumerate(
    ParameterSampler(param_dist, n_iter=N_TRIALS, random_state=42), start=1
):
    params = {
        **sampled,
        "n_estimators": MAX_TREES,   # capped by early stopping
        "subsample_freq": 1,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    mean_auc, mean_iter = evaluate_params(params)
    results.append((mean_auc, params, mean_iter))
    print(f"trial {trial_id}/{N_TRIALS}: cv AUC = {mean_auc:.4f}  (best_iter ~ {mean_iter:.0f})")

best_cv_auc, best_params, best_num_trees = max(results, key=lambda r: r[0])
print(f"\nBest CV AUC: {best_cv_auc:.4f}")
print("Best params:", {k: v for k, v in best_params.items() if k != "n_estimators"})

/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval

trial 1/30: cv AUC = 0.9080  (best_iter ~ 125)


/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval

trial 2/30: cv AUC = 0.9118  (best_iter ~ 122)


/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval

...

trial 28/30: cv AUC = 0.9166  (best_iter ~ 665)


/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval

trial 29/30: cv AUC = 0.9031  (best_iter ~ 298)


/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/Users/alex/IEEE-CIS_Fraud_Detection_MLOp/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval

trial 30/30: cv AUC = 0.9101  (best_iter ~ 834)

Best CV AUC: 0.9195
Best params: {'colsample_bytree': np.float64(0.5175897174514872), 'learning_rate': np.float64(0.011097554561103107), 'min_child_samples': 49, 'num_leaves': 220, 'reg_alpha': np.float64(0.00877781550471966), 'reg_lambda': np.float64(0.002273762810253686), 'subsample': np.float64(0.9143687545759647), 'subsample_freq': 1, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


In [8]:
# --- Retrain the best config on all of X_train, evaluate on X_val ---------
# Use the tree budget found by early stopping during the best trial (not 2000).
final_params = {**best_params, "n_estimators": int(round(best_num_trees))}

with mlflow.start_run(run_name="finetuned_lgbm"):
    mlflow.log_params(final_params)

    model = lgb.LGBMClassifier(**final_params)
    model.fit(X_train, y_train)

    y_score = model.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, y_score=y_score)  # kwarg is y_score in sklearn 1.9.0
    mlflow.log_metric("val_auc", val_auc)

    print(f"finetuned_lgbm: val AUC = {val_auc:.4f}")

finetuned_lgbm: val AUC = 0.9210


## Compare with the baseline

`baseline_lgbm` lives in the `ieee-fraud-detection` experiment, `finetuned_lgbm`
in `ieee-fraud-detection-finetune` — so the compare pulls the best `val_auc`
per run name from **both** experiments. Both were scored on the same `X_val`,
so the difference is the real gain from tuning.

In [9]:
# --- Compare with the transaction-only baseline ---------------------------
from mlflow.tracking import MlflowClient

client = MlflowClient()
exp_ids = [
    e.experiment_id
    for e in client.search_experiments()
    if e.name in {"ieee-fraud-detection", "ieee-fraud-detection-finetune"}
]


def best_auc(run_name: str) -> float:
    best = float("-inf")
    for exp_id in exp_ids:
        for r in client.search_runs([exp_id]):
            if r.data.tags.get("mlflow.runName", "?") == run_name:
                auc = r.data.metrics.get("val_auc", float("nan"))
                if auc == auc:  # skip NaN
                    best = max(best, auc)
    return best


base_auc = best_auc("baseline_lgbm")
tuned_auc = best_auc("finetuned_lgbm")
print(f"baseline_lgbm:  {base_auc:.4f}")
print(f"finetuned_lgbm: {tuned_auc:.4f}")
print(f"improvement:    {tuned_auc - base_auc:+.4f}")

baseline_lgbm:  0.9006
finetuned_lgbm: 0.9210
improvement:    +0.0205
